# 🎬 LeLe Chinese Quiz Video Rendering & QC Engine (Google Colab)

Welcome to the **LeLe Chinese Quiz Production Engine** for Google Colab.
This notebook enables GPU-accelerated rendering and automated Quality Control (QC) for all 4 quiz tabs:
- `pinyin` (Pinyin Tone Quiz)
- `vocabCN` (Chinese Vocabulary Quiz)
- `vocabVN` (Vietnamese-to-Chinese Quiz)
- `multilevels` (HSK 1-6 Progressive Quiz)

---

### 🚀 Key Features & Pipeline Standards
1. **High-Fidelity Rendering**: Manim Community Edition 1080x1920 (9:16 vertical video) at 60 fps.
2. **Natural Neural TTS**: Microsoft Edge-TTS with multi-speaker Chinese (`zh-CN-XiaoxiaoNeural`, `zh-CN-YunxiNeural`) and Vietnamese (`vi-VN-HoaiMyNeural`).
3. **Automated Gatekeeper 2 QC**: Automated physical video & audio verification before publishing eligibility.
4. **Direct Playable Links**: Video uploads write direct streamable URLs to **Column K** (`https://drive.google.com/file/d/{FILE_ID}/view?usp=drivesdk`).
5. **Zero-Leak Vault**: Secure credential isolation with `chmod 600` file protection.
6. **21px Row Height Invariant**: Strict layout consistency enforced across 100% of rows in all 4 tabs.

---

### ⚙️ Hardware Acceleration Setup
For fastest render speeds, enable an Nvidia GPU:
1. Go to **Runtime** (Thời gian chạy) menu -> **Change runtime type** (Thay đổi loại thời gian chạy).
2. Under **Hardware accelerator** (Trình tăng tốc phần cứng), select **T4 GPU**.
3. Click **Save**.
*(If GPU quotas are exceeded, the pipeline automatically falls back to High-Speed CPU execution).*


In [ ]:
#@title ⚡ 1. Hardware Detection & Accelerator Check
#@markdown Inspects available GPU accelerator (Nvidia T4/L4) or CPU fallback.

import subprocess, sys

try:
    gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if gpu_info.returncode == 0:
        print("✓ NVIDIA GPU Accelerator Active:\n")
        for line in gpu_info.stdout.splitlines()[:12]:
            print(f"  {line}")
    else:
        print("⚡ GPU not available. Running in High-Speed CPU mode (Automatic Fallback Active).")
except Exception as e:
    print(f"⚡ Running in CPU fallback mode: {e}")

try:
    import torch
    if torch.cuda.is_available():
        print(f"\n🚀 CUDA Device: {torch.cuda.get_device_name(0)}")
        print(f"🚀 Device Count: {torch.cuda.device_count()}")
        print(f"🚀 Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
    else:
        print("⚡ PyTorch CUDA not active - utilizing multi-core CPU threading.")
except Exception:
    pass


In [ ]:
#@title 📦 2. Install System Dependencies (FFmpeg, Cairo, Pango, CJK Fonts)
#@markdown Installs C/C++ graphics libraries, audio codecs, and full CJK + Emoji font coverage.
!apt-get update -qq
!apt-get install -y -qq ffmpeg libcairo2-dev libpango1.0-dev pkg-config fonts-noto-cjk fonts-noto-color-emoji
print("✓ System dependencies installed successfully: ffmpeg, cairo, pango, fonts-noto-cjk, fonts-noto-color-emoji.")


In [ ]:
#@title 🐍 3. Install Python Dependencies (100% GitHub Actions Parity)
#@markdown Installs the complete 16-package reference dependency set matching GitHub Actions Workflow 02.
!pip install --upgrade pip -q
!pip install --break-system-packages -q \
    manim \
    edge-tts \
    opencv-python-headless \
    opencc-python-reimplemented \
    gspread \
    google-api-python-client \
    google-auth \
    google-auth-oauthlib \
    pypinyin \
    requests \
    rich \
    scipy \
    numpy \
    Pillow \
    pydub \
    pyyaml

import importlib
required_16 = {
    "manim": "manim",
    "edge_tts": "edge-tts",
    "cv2": "opencv-python-headless",
    "opencc": "opencc-python-reimplemented",
    "gspread": "gspread",
    "googleapiclient": "google-api-python-client",
    "google.auth": "google-auth",
    "google_auth_oauthlib": "google-auth-oauthlib",
    "pypinyin": "pypinyin",
    "requests": "requests",
    "rich": "rich",
    "scipy": "scipy",
    "numpy": "numpy",
    "PIL": "Pillow",
    "pydub": "pydub",
    "yaml": "pyyaml"
}

print("\n🔍 Verifying 16/16 Dependency Parity against GitHub Actions:")
missing = []
for mod, pkg in required_16.items():
    try:
        importlib.import_module(mod)
        print(f"  ✓ {pkg:<30} ({mod})")
    except ImportError as e:
        print(f"  ❌ {pkg:<30} (ImportError: {e})")
        missing.append(pkg)

if not missing:
    print("\n🎉 [PARITY_VERIFIED] All 16/16 dependencies are installed with 100% GHA parity!")
else:
    raise RuntimeError(f"Missing dependencies: {missing}")


In [ ]:
#@title 📂 4. Codebase & Font Setup
#@markdown Prepares repository directory, extracts code bundle if uploaded, and configures fonts.
import os, sys, shutil, subprocess
from pathlib import Path

content_dir = Path("/content")
quiz_dir = content_dir / "quiz"
quiz_dir.mkdir(parents=True, exist_ok=True)

# 1. Extract code bundle if uploaded as quiz_bundle.tar.gz
bundle_file = content_dir / "quiz_bundle.tar.gz"
if bundle_file.exists():
    print(f"📦 Extracting codebase bundle from {bundle_file} into {quiz_dir}...")
    subprocess.run(["tar", "-xzf", str(bundle_file), "-C", str(quiz_dir)], check=True)
    print("✓ Codebase bundle extracted successfully.")
else:
    print(f"ℹ️ Codebase directory initialized at {quiz_dir}.")

# 2. Configure fonts in ~/.fonts
fonts_dir = Path.home() / ".fonts"
fonts_dir.mkdir(parents=True, exist_ok=True)

for sub in ["pinyinquiz", "vocabCNquiz", "vocabVNquiz", "multilevelsquiz"]:
    sub_fonts = quiz_dir / sub / "assets" / "fonts"
    if sub_fonts.exists():
        for f in sub_fonts.glob("*.*"):
            dest = fonts_dir / f.name
            if not dest.exists():
                try:
                    shutil.copy2(f, dest)
                except Exception:
                    pass

subprocess.run(["fc-cache", "-f"], check=False)
print("✓ Font cache refreshed with fontconfig.")


In [ ]:
#@title 🔑 5. Zero-Leak Vault Credentials Setup
#@markdown Securely registers Google Service Account and OAuth 2.0 credentials with restricted file permissions (chmod 600).
import os, json, shutil
from pathlib import Path

vault_base = Path.home() / ".cloud-profiles" / "lelehoctiengtrung"
sa_vault = vault_base / "google_sa"
oauth_vault = vault_base / "google_oauth"
telegram_vault = vault_base / "telegram"

sa_vault.mkdir(parents=True, exist_ok=True)
oauth_vault.mkdir(parents=True, exist_ok=True)
telegram_vault.mkdir(parents=True, exist_ok=True)

search_locations = [
    Path("/content"),
    Path("/content/drive/MyDrive"),
    Path.cwd(),
    Path("/content/quiz")
]

def locate_credential(filename):
    for loc in search_locations:
        candidate = loc / filename
        if candidate.exists() and candidate.is_file():
            return candidate
    return None

# 1. Service Account setup
sa_src = locate_credential("service_account.json")
if sa_src:
    target_sa = sa_vault / "service_account.json"
    shutil.copy2(sa_src, target_sa)
    os.chmod(target_sa, 0o600)
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(target_sa)
    print(f"✓ Service Account credentials linked from {sa_src} (chmod 600 locked).")
else:
    print("ℹ️ service_account.json not found in search paths. Upload if Google Sheets API access is needed.")

# 2. OAuth 2.0 User Credentials setup
oauth_src = locate_credential("user_oauth2.json") or locate_credential("oauth_credentials.json")
if oauth_src:
    target_oauth = oauth_vault / "user_oauth2.json"
    shutil.copy2(oauth_src, target_oauth)
    os.chmod(target_oauth, 0o600)
    print(f"✓ Google User OAuth 2.0 credentials linked from {oauth_src} (chmod 600 locked).")
else:
    print("ℹ️ user_oauth2.json not found in search paths. Upload if Google Drive uploads are needed.")


In [ ]:
#@title 🎬 6. Interactive Video Rendering & Gatekeeper 2 QC Harness
#@markdown Configure target tab, specific row ID (leave blank for all pending rows), and render quality.

TARGET_TAB = "all" #@param ["all", "pinyin", "vocabCN", "vocabVN", "multilevels"]
ROW_ID = "" #@param {type:"string"}
QUALITY = "qh" #@param ["qh", "ql", "qm", "qk"]

import os, sys, subprocess
from pathlib import Path

quiz_dir = Path("/content/quiz")
if not quiz_dir.exists() or not (quiz_dir / "scripts").exists():
    quiz_dir = Path.cwd()

env = os.environ.copy()
env["PYTHONPATH"] = f"{quiz_dir}:{env.get('PYTHONPATH', '')}"

dispatcher_script = quiz_dir / "scripts" / "run_render_dispatcher.py"
if not dispatcher_script.exists():
    dispatcher_script = Path("scripts/run_render_dispatcher.py")

cmd = [
    sys.executable, str(dispatcher_script),
    "--tab", TARGET_TAB,
    "--quality", QUALITY,
    "--local"
]

clean_row = ROW_ID.replace("#", "").strip()
if clean_row:
    cmd.extend(["--row", clean_row])

print(f"🚀 Launching Render Dispatcher:")
print(f"   Command: {' '.join(cmd)}\n")

proc = subprocess.Popen(
    cmd, cwd=str(quiz_dir), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

if proc.stdout:
    for line in proc.stdout:
        print(line, end="", flush=True)

proc.wait()

if proc.returncode == 0:
    print("\n🎉 [RENDER_AND_QC_SUCCESS] Video rendering, GDrive upload, and Gatekeeper 2 QC completed successfully!")
else:
    print(f"\n❌ [RENDER_ERROR] Render dispatcher exited with return code {proc.returncode}")


In [ ]:
#@title 📏 7. Enforce 21px Row Height Invariant
#@markdown Audits and locks 100% of rows across all 4 quiz tabs to strict 21px row height standard.
import os, sys, subprocess
from pathlib import Path

quiz_dir = Path("/content/quiz")
if not quiz_dir.exists() or not (quiz_dir / "scripts").exists():
    quiz_dir = Path.cwd()

enforce_script = quiz_dir / "scripts" / "enforce_row_height_21px.py"
if not enforce_script.exists():
    enforce_script = Path("scripts/enforce_row_height_21px.py")

print("📏 Executing 21px Row Height Invariant Enforcer...")
res = subprocess.run([sys.executable, str(enforce_script)], cwd=str(quiz_dir))
if res.returncode == 0:
    print("✓ 21px Row Height Invariant verified and locked across all 4 quiz worksheets.")
else:
    print(f"⚠️ Row height enforcer exited with code {res.returncode}")
